# Machine Learning for Clinical Variant Interpretation — Portfolio Walkthrough

**Predicting CADD deleteriousness scores and ClinVar interpretation conflicts with explainable machine learning**

This notebook is a condensed walkthrough of the full analysis for interview/portfolio review. It loads **already-computed results** from `results/` — it does not retrain any models (the full tuning pipeline takes ~140 minutes; see `docs/workflow.md` to reproduce it).

> This is a research/portfolio project, not a clinical variant-classification system. `CADD_PHRED` is a deleteriousness-prioritization score, not an ACMG/AMP pathogenicity classification. `CLASS` is ClinVar submitter interpretation-conflict, not a pathogenic/benign label. See `docs/limitations.md`.

**Dataset**: [ClinVar Conflicting Classifications](https://www.kaggle.com/datasets/kevinarvai/clinvar-conflicting) (Kaggle) — 65,188 variants x 46 columns.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
DATA_RAW = ROOT / "data" / "raw" / "clinvar_conflicting.csv"

df = pd.read_csv(DATA_RAW, low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nCADD_PHRED (regression target) missing: {df['CADD_PHRED'].isna().mean()*100:.2f}%")
display(df["CADD_PHRED"].describe())
print(f"\nCLASS (conflict, secondary target) distribution:")
display(df["CLASS"].value_counts(normalize=True).rename("share"))

## 1. Regression: predicting CADD_PHRED

Three model families (Linear Regression, Random Forest, XGBoost) were compared, then XGBoost was carried through feature engineering (log-transformed allele frequency, parsed position fields) and hyperparameter tuning (`RandomizedSearchCV`). All evaluated on the same 80/20 holdout split (`random_state=42`).

**Key insight**: feature engineering contributed more than hyperparameter tuning (R2 +0.021 vs. +0.004) — domain-informed features beat additional search here.

In [ ]:
holdout = pd.read_csv(RESULTS / "regression" / "model_improvement" / "holdout_comparison.csv", index_col=0)
display(holdout.round(3))

eng_gain = holdout.loc["engineered_XGBoost", "R2"] - holdout.loc["baseline_XGBoost", "R2"]
tune_gain = holdout.loc["tuned_XGBoost", "R2"] - holdout.loc["engineered_XGBoost", "R2"]
print(f"\nFeature engineering gain (XGBoost R2): +{eng_gain:.3f}")
print(f"Hyperparameter tuning gain (XGBoost R2):  +{tune_gain:.3f}")

display(Image(filename=str(RESULTS / "regression" / "model_comparison.png")))

## 2. SHAP interpretation

SHAP (`TreeExplainer`) on the tuned XGBoost regressor, aggregated from one-hot columns back to original variables. Top result: `IMPACT > SIFT > PolyPhen > Consequence`.

In [ ]:
shap_imp = pd.read_csv(RESULTS / "explainability" / "shap" / "feature_importance_by_variable.csv", index_col=0)
display(shap_imp.head(10).round(3))
display(Image(filename=str(RESULTS / "explainability" / "shap" / "01_importance_by_variable.png")))

## 3. From SHAP to ablation: is SIFT/PolyPhen importance genuine?

SIFT and PolyPhen both estimate protein-functional damage — conceptually overlapping with CADD itself, raising a redundancy hypothesis. Retraining the tuned XGBoost **without** SIFT/PolyPhen tests this directly.

In [ ]:
ablation = pd.read_csv(RESULTS / "explainability" / "ablation" / "ablation_comparison.csv", index_col=0)
display(ablation.round(3))

full_r2 = ablation.loc["full (SIFT+PolyPhen 포함)", "R2"]
ablated_r2 = ablation.loc["ablation (SIFT+PolyPhen 제외)", "R2"]
print(f"\nR2 drop from removing SIFT+PolyPhen: {full_r2:.3f} -> {ablated_r2:.3f} "
      f"({(full_r2-ablated_r2)/full_r2*100:.1f}% relative decrease)")
print("Conclusion: SHAP importance alone does NOT establish redundancy - SIFT/PolyPhen carry")
print("substantial independent signal that IMPACT/Consequence/BLOSUM62 do not capture.")

display(Image(filename=str(RESULTS / "explainability" / "ablation" / "ablation_comparison.png")))

## 4. Generalization: does the model work on genes it has never seen?

The standard random 80/20 split lets the same gene appear in both train and test (95.7% of test-set genes also appear in training). A gene-based `GroupShuffleSplit`/`GroupKFold` (grouped by `SYMBOL`) tests strictly unseen-gene performance.

In [ ]:
split_cmp = pd.read_csv(RESULTS / "generalization" / "split_comparison.csv", index_col=0)
display(split_cmp.round(3))

gkf = pd.read_csv(RESULTS / "generalization" / "groupkfold_summary.csv", index_col=0)
print("GroupKFold 5-fold summary:")
display(gkf.round(3))

display(Image(filename=str(RESULTS / "generalization" / "split_comparison.png")))
print("\nInterpretation: R2 fell from ~0.71 (random split) to ~0.64 (unseen genes) - the random")
print("split was optimistic, but real signal remains beyond gene memorization.")

## 5. Residual investigation: the 23-36 "band" artifact

The residual plot showed a dense vertical band between true CADD_PHRED 23-36. Root-cause analysis (`src/11_residual_investigation.py`) found this is **mostly a quantization/overplotting artifact** (CADD_PHRED values are heavily rounded in this range - e.g. 34.0 appears 1,431 times), not evidence of worse model performance there. A smaller, real shrinkage bias and a SIFT/PolyPhen-discordance failure mode remain (see `docs/model_validation.md`, `docs/variant_case_studies.md`).

In [ ]:
display(Image(filename=str(RESULTS / "regression" / "01_residual_plot.png")))

band_errors = pd.read_csv(RESULTS / "biological_insights" / "residual_investigation" / "band_top_errors.csv")
print("Largest residual cases inside the 23-36 band (top 5):")
display(band_errors[["SYMBOL", "Consequence", "IMPACT", "SIFT", "PolyPhen", "CADD_PHRED", "pred", "residual"]].head(5))

## 6. Secondary task: ClinVar interpretation-conflict classification

Predicting `CLASS` (submitter disagreement) from the same engineered features + `CADD_PHRED`. This is a harder, noisier problem than the regression task - ROC-AUC 0.791 vs. regression R2≈0.71 - consistent with conflict status depending partly on factors (submitter criteria, evidence completeness, phenotype context) this dataset does not capture.

In [ ]:
clf_cmp = pd.read_csv(RESULTS / "classification" / "classification_comparison.csv", index_col=0)
display(clf_cmp.round(3))
display(Image(filename=str(RESULTS / "classification" / "roc_curves.png")))

thresholds = pd.read_csv(RESULTS / "classification" / "threshold_analysis" / "threshold_scenarios.csv")
print("\nPrecision-Recall tradeoff at different thresholds:")
display(thresholds.round(3))

## 7. Limitations

- `CADD_PHRED` != ACMG/AMP clinical pathogenicity classification.
- SIFT/PolyPhen/IMPACT/Consequence/CADD are correlated annotations - some information overlap is inherent to the feature set (mitigated, not eliminated, by the ablation check above).
- The random split is optimistic; Gene Group Split is stricter but still not full external validation.
- `CLASS` reflects submitter disagreement, not verified ground-truth pathogenicity.
- No prospective clinical validation was performed.

Full list: [`docs/limitations.md`](../docs/limitations.md).

## 8. Key takeaways

1. Genomic/functional-annotation features explain a substantial share of CADD_PHRED variance (R2≈0.71 on a random holdout; R2≈0.64-0.65 on strictly unseen genes).
2. Feature engineering (domain-informed transforms) beat hyperparameter tuning for this dataset.
3. SHAP importance is not the same as feature necessity - only the ablation experiment confirmed SIFT/PolyPhen's independent contribution.
4. Evaluation methodology matters: the random split's 95.7% gene overlap made results look better than they would generalize to truly new genes.
5. Predicting ClinVar interpretation conflict is a harder, noisier problem than predicting deleteriousness - a reminder that not everything genomic features can explain.

See [`docs/analysis_report.md`](../docs/analysis_report.md) for the full 11-section report, and [`README.md`](../README.md) for the project landing page.